# Call the C-MAPSS batch endpoint

## 1. Configure the endpoint

No subscription ID, resource group, workspace configuration, or AML SDK client is needed. Authentication uses your existing **Azure CLI login**. If needed, run `az cloud set --name AzureUSGovernment` and then `az login` in a terminal.

Set `INPUT_URI` below to the Azure storage/datastore URI of an already uploaded CSV. To upload the two-row test file once, use AML Studio **Data > Create > File (uri_file)** and select `models/cmapss-fd001-sklearn15/batch-test.csv`. Copy the resulting asset's **data path** (a datastore URI or HTTPS storage URL), not the local path or the Studio page URL. The CSV must contain all 25 model input columns.

The URL matches your screenshot. No deployment header is sent, so the endpoint uses its current default deployment (`cmapss-fd001-local-regressor-2` in the screenshot).

This is an **asynchronous batch API**: the response contains a job name, not predictions. Running the POST can start billable compute. This notebook does not upload files or submit a job until you run the POST cell.

Source: A. Saxena and K. Goebel (2008), NASA C-MAPSS. Simulation predictions are not certified maintenance advice.

In [2]:
import json
import uuid
from urllib.parse import urlsplit

import requests
from azure.identity import AzureCliCredential

ENDPOINT_URL = "https://aml-demo-turbine-local.usgovvirginia.inference.ml.azure.us/jobs"
INPUT_URI = ""

credential = AzureCliCredential()
endpoint_url = ENDPOINT_URL
http_method = "POST"
query_parameters = {}
print("URL:", endpoint_url)
print("Authentication: your Azure CLI login (AzureUSGovernment)")

URL: https://aml-demo-turbine-local.usgovvirginia.inference.ml.azure.us/jobs
Authentication: your Azure CLI login (AzureUSGovernment)


In [3]:
input_uri = INPUT_URI.strip()
if not input_uri:
    raise ValueError("Set INPUT_URI to the uploaded CSV's Azure datastore URI or HTTPS storage URL.")
parsed_uri = urlsplit(input_uri)
if parsed_uri.scheme not in {"azureml", "https"} or not parsed_uri.netloc:
    raise ValueError("Use an azureml:// datastore URI or https:// storage URL, not a local file path.")
request_body = {
    "properties": {
        "InputData": {
            "UriFile": {"JobInputType": "UriFile", "Uri": input_uri}
        }
    }
}
request_headers = {
    "Content-Type": "application/json",
    "Repeatability-Request-ID": str(uuid.uuid4()),
}
print(http_method, endpoint_url)
print(json.dumps({**request_headers, "Authorization": "Bearer <redacted>"}, indent=2))
print(json.dumps(request_body, indent=2))

ValueError: Set INPUT_URI to the uploaded CSV's Azure datastore URI or HTTPS storage URL.

## 2. Send the API request

The previous cell only previews the request. Run the next cell **once** to start the batch job. It obtains a Government ML token from your Azure CLI login, never prints the token, and does not automatically retry POST requests. If the connection times out, check the endpoint's Jobs tab before submitting again; the service may already have accepted the job.

In [ ]:
response = None
job_name = None
token = credential.get_token("https://ml.azure.us/.default")
try:
    response = requests.request(
        method=http_method,
        url=endpoint_url,
        params=query_parameters,
        headers={**request_headers, "Authorization": f"Bearer {token.token}"},
        json=request_body,
        timeout=120,
        allow_redirects=False,
    )
except (requests.ConnectionError, requests.Timeout):
    raise RuntimeError("No response received. Check Azure's Jobs tab before resubmitting; a job may already exist.") from None
finally:
    del token
    if response is not None:
        response.request.headers["Authorization"] = "Bearer <redacted>"

## 3. Display the API response

The next cell displays the actual HTTP status and response body, including errors. A successful response contains the asynchronous job name. In AML Studio, open this endpoint's **Jobs** tab and select that job to check its status, logs, and output `predictions.csv`. No additional workspace settings are needed for this notebook.

In [ ]:
if response is None:
    raise RuntimeError("Run the submit cell first and check for connection errors.")
print("HTTP", response.status_code)
try:
    response_body = response.json()
    print(json.dumps(response_body, indent=2))
except ValueError:
    response_body = None
    print(response.text)
response.raise_for_status()
if not 200 <= response.status_code < 300:
    raise RuntimeError("Unexpected non-success response; inspect the body above.")
if not isinstance(response_body, dict) or not response_body.get("name"):
    raise RuntimeError("Expected a job name in the response; inspect the body above.")
job_name = response_body["name"]
print("Job name:", job_name)